### EOF analysis

In [ ]:
#!/usr/bin/env python3
# ============================================================
# Combined EOF Analysis for CMIP6 ocean variables in (lat,depth)
# at fixed longitudes, INCLUDING South + North Atlantic
#
# EOFs fit on TRAIN only; PCs are projected for FULL period.
# ============================================================

import os
import glob
import re
import numpy as np
import xarray as xr
from eofs.standard import Eof
import warnings
warnings.filterwarnings("ignore")

# Prevent HDF5 file locking issues on shared filesystems
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# ============================================================
# CONFIG
# ============================================================
MODEL   = "IPSL-CM6A-LR"      # EC-Earth3, IPSL-CM6A-LR, CESM2, MPI-ESM1-2-LR
VAR     = "so"             # thetao or so
NMODES  = 10

START_YEAR = 1850
END_YEAR   = 2014

TARGET_LONS   = [-10.0, -20.0, -30.0, -40.0, -60.0]
LON_WINDOW    = 10.0       # degrees
DEPTH_RANGE_M = (0.0, 5000.0)

# >>> NEW: Train fraction of the COMMON time axis (first X%)
TRAIN_FRACTION = 0.85  # choose 0.80–0.85 as you like

# ============================================================
# DIRECTORIES
# ============================================================
if MODEL == "EC-Earth3":
    IN_DIR = f"/data/projects/nckf/frekle/CMIP6_data/EC-Earth3/{VAR}/masked/"
elif MODEL == "IPSL-CM6A-LR":
    IN_DIR = f"/data/projects/nckf/frekle/CMIP6_data/IPSL-CM6A-LR/{VAR}/masked/"
elif MODEL == "CESM2":
    IN_DIR = f"/data/projects/nckf/frekle/CMIP6_data/CESM2/{VAR}/masked/"
elif MODEL == "MPI-ESM1-2-LR":
    IN_DIR = f"/data/projects/nckf/frekle/CMIP6_data/MPI-ESM1-2-LR/{VAR}/masked/"
else:
    raise ValueError("Unknown MODEL")

OUT_DIR = f"/data/projects/nckf/frekle/EOF_results/{MODEL}/latdepth_sections/Train_period_{int(TRAIN_FRACTION*100)}pct/"
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# UTILITIES
# ============================================================

def parse_member_label(path):
    m = re.search(r"_(r\d+i\d+p\d+f\d+)", os.path.basename(path))
    return m.group(1) if m else os.path.basename(path)

def wrap_lon(lon):
    return ((lon + 180) % 360) - 180

def standardize_latlon(ds):
    lat_var = next(v for v in ["lat", "latitude", "nav_lat"] if v in ds)
    lon_var = next(v for v in ["lon", "longitude", "nav_lon"] if v in ds)

    lat = ds[lat_var]
    lon = ds[lon_var]

    if lat.ndim == 1 and lon.ndim == 1:
        lon2d, lat2d = np.meshgrid(lon.values, lat.values)
        ds = ds.drop_vars([lat_var, lon_var])
        ds = ds.rename_dims({lat.dims[0]: "y", lon.dims[0]: "x"})
        ds["lat2d"] = xr.DataArray(lat2d, dims=("y", "x"))
        ds["lon2d"] = xr.DataArray(lon2d, dims=("y", "x"))
        return ds

    ds = ds.swap_dims({lat.dims[0]: "y", lat.dims[1]: "x"})
    ds = ds.rename({lat_var: "lat2d", lon_var: "lon2d"})
    return ds

def get_depth_dim_and_coord(da, ds):
    if "olevel" in da.dims:
        return "olevel", ds["olevel"].values
    if "depth" in da.dims:
        return "depth", ds["depth"].values
    if "lev" in da.dims:
        z = ds["lev"].values
        if np.nanmax(z) > 10000:
            z = z / 100.0
        return "lev", z
    raise KeyError("No vertical coordinate found")

def compute_layer_thickness(z):
    z = np.asarray(z)
    dz = np.empty_like(z, dtype=float)
    dz[1:-1] = 0.5 * (z[2:] - z[:-2])
    dz[0]    = z[1] - z[0]
    dz[-1]   = z[-1] - z[-2]
    return np.abs(dz)

def extract_section_valid_ocean(da, ds, target_lon, depth_dim):
    lon2d = ds["lon2d"].values
    lat2d = ds["lat2d"].values

    lonW = wrap_lon(lon2d)
    tW   = wrap_lon(target_lon)

    valid_xy = np.isfinite(da).any(dim=("time", depth_dim)).values
    ny, nx = lonW.shape

    ix = np.full(ny, -1, dtype=int)

    for y in range(ny):
        cand = np.where(valid_xy[y])[0]
        if cand.size == 0:
            continue

        d = np.abs(lonW[y, cand] - tW)
        ok = d <= LON_WINDOW
        if not np.any(ok):
            continue

        cand = cand[ok]
        d    = d[ok]
        ix[y] = cand[np.argmin(d)]

    valid_row = ix >= 0

    sec = da.isel(x=xr.DataArray(np.where(valid_row, ix, 0), dims="y"))
    sec = sec.where(xr.DataArray(valid_row, dims="y"))

    lat_1d = np.where(valid_row, lat2d[np.arange(ny), ix], np.nan)

    return sec, lat_1d, valid_row

# ============================================================
# MAIN WORKFLOW
# ============================================================

def run_longitude(target_lon):

    print("\n" + "="*80)
    print(f"▶ Longitude {target_lon:.1f}° | MODEL={MODEL} | VAR={VAR}")
    print("="*80)

    files = sorted(glob.glob(os.path.join(IN_DIR, f"*{VAR}*_masked*.nc")))
    if not files:
        raise FileNotFoundError("No masked files found")

    sections = []
    member_labels = []

    lat_ref = z_ref = dz_ref = depth_dim_ref = None

    for i, f in enumerate(files, start=1):
        label = parse_member_label(f)
        member_labels.append(label)

        print(f"\n  → Member {i}/{len(files)}: {label}")

        ds = standardize_latlon(xr.open_dataset(f))
        da = ds[VAR]

        depth_dim, z = get_depth_dim_and_coord(da, ds)
        dz = compute_layer_thickness(z)

        da = da.sel(time=slice(f"{START_YEAR}-01-01", f"{END_YEAR}-12-31"))

        sec, lat1d, valid = extract_section_valid_ocean(
            da, ds, target_lon, depth_dim
        )

        keepz = (z >= DEPTH_RANGE_M[0]) & (z <= DEPTH_RANGE_M[1])
        sec = sec.isel({depth_dim: keepz})
        z_sel  = z[keepz]
        dz_sel = dz[keepz]

        good = np.isfinite(lat1d)
        sec = sec.isel(y=good)
        lat1d = lat1d[good]

        order = np.argsort(lat1d)
        sec = sec.isel(y=order)
        lat1d = lat1d[order]

        # anomaly over the full period; TRAIN solver will still be fit only on TRAIN time
        sec = sec - sec.mean("time")
        sec = sec.expand_dims(member=[label])
        sections.append(sec)

        if lat_ref is None:
            lat_ref = lat1d
            z_ref   = z_sel
            dz_ref  = dz_sel
            depth_dim_ref = depth_dim

        print(f"    Kept latitude rows: {lat1d.size}")

    print("\n▶ Aligning time across members")
    common_time = sections[0]["time"].values
    for s in sections[1:]:
        common_time = np.intersect1d(common_time, s["time"].values)

    # >>> NEW: ensure chronological order
    common_time = np.sort(common_time)

    combined = xr.concat([s.sel(time=common_time) for s in sections], dim="member")

    # SAFE renaming: avoid 'lat' name collision
    combined = combined.rename({
        "y": "lat_index",
        depth_dim_ref: "depth"
    })

    combined = combined.assign_coords(
        lat=("lat_index", lat_ref),
        depth=("depth", z_ref),
    )

    n_member = combined.sizes["member"]
    n_time   = combined.sizes["time"]
    n_lat    = combined.sizes["lat_index"]
    n_depth  = combined.sizes["depth"]

    # >>> NEW: define TRAIN time split on the common axis
    n_train = int(np.floor(TRAIN_FRACTION * n_time))
    n_train = max(2, min(n_train, n_time))  # safety
    train_time = combined["time"].values[:n_train]

    train_mask = np.zeros(n_time, dtype=bool)
    train_mask[:n_train] = True

    print(f"\n▶ Combined shape: member={n_member}, time={n_time}, lat={n_lat}, depth={n_depth}")
    print(f"▶ TRAIN_FRACTION={TRAIN_FRACTION} -> n_train={n_train}/{n_time} "
          f"(train end = {str(train_time[-1])})")

    # FULL data (for projection)
    data_full = combined.transpose("member", "time", "lat_index", "depth").values
    data2d_full = data_full.reshape(n_member * n_time, n_lat * n_depth)
    data2d_full = np.ma.masked_invalid(data2d_full)

    # TRAIN data (for fitting EOFs)
    combined_train = combined.sel(time=train_time)
    data_train = combined_train.transpose("member", "time", "lat_index", "depth").values
    data2d_train = data_train.reshape(n_member * n_train, n_lat * n_depth)
    data2d_train = np.ma.masked_invalid(data2d_train)

    print("▶ Computing EOF weights")
    w_lat = np.clip(np.cos(np.deg2rad(lat_ref)), 0, None)
    w2d   = np.sqrt(w_lat[:, None] * dz_ref[None, :])
    weights = w2d.reshape(n_lat * n_depth)

    print("▶ Solving EOFs on TRAIN only")
    solver = Eof(data2d_train, weights=weights)

    EOFs = solver.eofs(neofs=NMODES).reshape(NMODES, n_lat, n_depth)
    VF   = solver.varianceFraction()[:NMODES]

    # >>> NEW: project FULL period onto TRAIN EOFs
    # projectField returns (n_samples, neofs) where n_samples = n_member*n_time
    PCs_full_flat = solver.projectField(data2d_full, neofs=NMODES)
    PCs = np.asarray(PCs_full_flat).reshape(n_member, n_time, NMODES)

    tag = f"{abs(target_lon):04.1f}".replace(".", "p")
    hemi = "W" if target_lon < 0 else "E"
    outfile = os.path.join(OUT_DIR, f"EOF_latdepth_{VAR}_{hemi}{tag}.nc")

    print(f"▶ Writing {outfile}")

    xr.Dataset(
        {
            "EOF": (("mode", "lat_index", "depth"), EOFs),
            "PC":  (("member", "time", "mode"), PCs),
            "variance_fraction": (("mode",), VF),
            "train_mask": (("time",), train_mask),
        },
        coords={
            "mode": np.arange(1, NMODES + 1),
            "member": np.array(member_labels, dtype=object),
            "time": combined["time"].values,
            "lat": ("lat_index", lat_ref),
            "depth": ("depth", z_ref),
        },
        attrs={
            "MODEL": MODEL,
            "VAR": VAR,
            "TARGET_LON_DEG": target_lon,
            "LON_WINDOW_DEG": LON_WINDOW,
            "WEIGHTING": "sqrt(cos(lat) * dz)",
            "DEPTH_RANGE_M": f"{DEPTH_RANGE_M[0]}-{DEPTH_RANGE_M[1]}",
            "EOF_FIT": "TRAIN_ONLY",
            "TRAIN_FRACTION": float(TRAIN_FRACTION),
            "TRAIN_START": str(train_time[0]),
            "TRAIN_END": str(train_time[-1]),
        }
    ).to_netcdf(outfile)

    print(f"✓ Done longitude {target_lon:.1f}°")

# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    print(f"\n=== MODEL={MODEL} | VAR={VAR} ===")
    print(f"TARGET_LONS={TARGET_LONS}")
    print(f"LON_WINDOW={LON_WINDOW} deg")
    print(f"DEPTH_RANGE_M={DEPTH_RANGE_M}")
    print(f"TRAIN_FRACTION={TRAIN_FRACTION}")

    for lon in TARGET_LONS:
        run_longitude(lon)

    print("\n✅ All longitudes processed successfully.")


### Feature selection

In [ ]:
#!/usr/bin/env python3
"""
MINIMAL FEATURE PREPARATION SCRIPT
=================================
This script produces:

(1) feature_ranking_pre.csv
    - correlation ranking computed on PRE period only
    - avoids post-period leakage

(2) final_features.csv
    - top-N ranked seed features expanded by ±W lags
    - useful as a broad candidate feature set for downstream analysis

Notes
-----
- PRE = years <= TRAIN_END_YEAR from EOF file attributes
- Ranking is done on PRE only

Required inputs
---------------
- EOF_latdepth_{var}_{lon}.nc files with PC variable
- AMOC_{MODEL}.nc with variable TARGET
"""

import os
import glob
import numpy as np
import pandas as pd
import xarray as xr

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# ============================================================
# USER SETTINGS
# ============================================================
MODEL   = "IPSL-CM6A-LR"   # "EC-Earth3", "IPSL-CM6A-LR", "CESM2", "MPI-ESM1-2-LR"
TARGET  = "AMOC_45N_member"

# Choose one:
MODE = "ensmean"
member_id = None

# Example for member mode:
# MODE = "member"
# member_id = "r1i1p1f1"

EOF_DIR   = f"/data/projects/nckf/frekle/EOF_results/{MODEL}/latdepth_sections/Train_period_85pct/"
AMOC_FILE = f"/data/users/frekle/AMOC_analysis/AMOC_{MODEL}.nc"

OUTDIR = f"/data/users/frekle/Final_figures/{MODEL}/{TARGET}/Feature_selection/"
os.makedirs(OUTDIR, exist_ok=True)

VARS     = ["thetao", "so"]
LON_TAGS = ["W10p0", "W20p0", "W30p0", "W40p0", "W60p0"]
N_MODES  = 10

YEAR_START = 1850
YEAR_END   = 2014

MAX_LAG_ALLOWED = 20
STANDARDIZE_PC = True

# Final feature expansion settings
N_SEEDS = 10   # number of top-ranked seed features
W       = 2    # expand each seed by ±W lags

# ============================================================
# Helper functions
# ============================================================

def extract_years(time_coord):
    """Extract integer years from a time coordinate."""
    try:
        return xr.DataArray(time_coord).dt.year.values.astype(int)
    except Exception:
        return np.array([int(str(x)[:4]) for x in np.asarray(time_coord)])


def corr_1d(a, b):
    """Pearson correlation for 1D arrays with NaN handling."""
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 3:
        return np.nan
    aa = a[m] - np.mean(a[m])
    bb = b[m] - np.mean(b[m])
    denom = np.sqrt(np.sum(aa**2) * np.sum(bb**2))
    if denom == 0:
        return np.nan
    return np.sum(aa * bb) / denom


def load_pc(var, lon, mode="ensmean", member_id=None):
    """Load PC array for one variable and longitude section."""
    f = os.path.join(EOF_DIR, f"EOF_latdepth_{var}_{lon}.nc")
    ds = xr.open_dataset(f)

    if "PC" not in ds:
        raise KeyError(f"'PC' not found in {f}")

    PC = ds["PC"].isel(mode=slice(0, N_MODES))

    if "member" in PC.dims:
        if mode == "ensmean":
            PC = PC.mean("member")
        elif mode == "member":
            if member_id is None:
                raise ValueError("member_id must be provided when mode='member'")
            if "member" in PC.coords and member_id in PC["member"].values:
                PC = PC.sel(member=member_id)
            else:
                raise ValueError(f"member_id {member_id} not found in {f}")
        else:
            raise ValueError("mode must be 'ensmean' or 'member'")

    PC = PC.transpose("time", "mode")
    years = extract_years(PC["time"])
    PC = PC.assign_coords(year=("time", years)).swap_dims({"time": "year"}).drop_vars("time")

    ds.close()
    return PC.astype(float)


def load_amoc(mode="ensmean", member_id=None):
    """Load AMOC target series."""
    ds = xr.open_dataset(AMOC_FILE)

    if TARGET not in ds:
        raise KeyError(f"Target '{TARGET}' not found in {AMOC_FILE}")

    y = ds[TARGET].squeeze()

    if "member" in y.dims:
        if mode == "ensmean":
            y = y.mean("member")
        elif mode == "member":
            if member_id is None:
                raise ValueError("member_id must be provided when mode='member'")
            if "member" in y.coords and member_id in y["member"].values:
                y = y.sel(member=member_id)
            else:
                raise ValueError(f"member_id {member_id} not found in AMOC file")
        else:
            raise ValueError("mode must be 'ensmean' or 'member'")

    if "year" not in y.dims:
        years = extract_years(y["time"])
        y = y.assign_coords(year=("time", years)).swap_dims({"time": "year"}).drop_vars("time")

    ds.close()
    return y.astype(float)


def get_train_end():
    """Read TRAIN_END year from the first EOF file found."""
    files = sorted(glob.glob(os.path.join(EOF_DIR, "EOF_latdepth_*_*.nc")))
    if not files:
        raise FileNotFoundError(f"No EOF files found in {EOF_DIR}")

    ds = xr.open_dataset(files[0])

    if "TRAIN_END" not in ds.attrs:
        ds.close()
        raise KeyError(f"TRAIN_END attribute not found in {files[0]}")

    y = int(str(ds.attrs["TRAIN_END"])[:4])
    ds.close()
    return y


def expand_seed_features(seed_features, max_lag_allowed=20, W=2):
    """
    Expand each seed feature by ±W lags.
    Removes duplicates while preserving order.
    """
    expanded = []
    seen = set()

    for var, lon, mode, lag in seed_features:
        for new_lag in range(max(0, lag - W), min(max_lag_allowed, lag + W) + 1):
            feat = (var, lon, mode, new_lag)
            if feat not in seen:
                expanded.append(feat)
                seen.add(feat)

    return expanded


# ============================================================
# Main workflow
# ============================================================

print("Loading data...")
TRAIN_END_YEAR = get_train_end()
print("PRE ends at:", TRAIN_END_YEAR)

amoc = load_amoc(mode=MODE, member_id=member_id)
pc_dict = {(v, l): load_pc(v, l, mode=MODE, member_id=member_id) for v in VARS for l in LON_TAGS}

# Align years
years = amoc["year"].values.astype(int)
for da in pc_dict.values():
    years = np.intersect1d(years, da["year"].values.astype(int))

years = years[(years >= YEAR_START) & (years <= YEAR_END)]
years.sort()

if len(years) == 0:
    raise ValueError("No overlapping years found after alignment.")

y = amoc.sel(year=years).values
PC = {k: v.sel(year=years).values for k, v in pc_dict.items()}

idx_pre = np.where(years <= TRAIN_END_YEAR)[0]
idx_post = np.where(years > TRAIN_END_YEAR)[0]

if len(idx_pre) == 0:
    raise ValueError("PRE period is empty.")
if len(idx_post) == 0:
    print("Warning: POST period is empty. That is okay for this script, since ranking only uses PRE.")

print(f"PRE:  {years[idx_pre[0]]} - {years[idx_pre[-1]]}")
if len(idx_post) > 0:
    print(f"POST: {years[idx_post[0]]} - {years[idx_post[-1]]}")

# ============================================================
# 1) Ranking on PRE
# ============================================================

print("Computing PRE ranking...")

rows = []
ypre = y[idx_pre]

for var in VARS:
    for lon in LON_TAGS:
        X = PC[(var, lon)][idx_pre, :]

        if STANDARDIZE_PC:
            X = (X - np.nanmean(X, axis=0)) / (np.nanstd(X, axis=0) + 1e-12)

        for lag in range(MAX_LAG_ALLOWED + 1):
            if lag >= len(ypre):
                continue

            t = np.arange(lag, len(ypre))

            for m0 in range(N_MODES):
                c = corr_1d(X[t - lag, m0], ypre[t])
                if np.isfinite(c):
                    rows.append({
                        "var": var,
                        "lon_tag": lon,
                        "mode": m0 + 1,     # saved 1-based
                        "lag": lag,
                        "corr": c,
                        "abs_corr": abs(c)
                    })

df_rank = pd.DataFrame(rows).sort_values("abs_corr", ascending=False).reset_index(drop=True)

rank_file = os.path.join(OUTDIR, "feature_ranking_pre.csv")
df_rank.to_csv(rank_file, index=False)
print(f"✅ Saved: {rank_file}")

# Convert ranking to tuples (mode back to 0-based internally)
feats_ranked = [
    (r.var, r.lon_tag, int(r.mode - 1), int(r.lag))
    for r in df_rank.itertuples(index=False)
]

# ============================================================
# 2) Build final_features.csv
# ============================================================

print("Building final feature list...")

seed_features = feats_ranked[:N_SEEDS]
final_features = expand_seed_features(
    seed_features,
    max_lag_allowed=MAX_LAG_ALLOWED,
    W=W
)

df_final = pd.DataFrame([
    {
        "var": var,
        "lon_tag": lon,
        "mode": mode + 1,   # save 1-based
        "lag": lag
    }
    for var, lon, mode, lag in final_features
])

final_file = os.path.join(OUTDIR, "final_features.csv")
df_final.to_csv(final_file, index=False)
print(f"✅ Saved: {final_file}")

# ============================================================
# Optional screen summary
# ============================================================

print("\nTop 10 ranked PRE features:")
print(df_rank.head(10).to_string(index=False))

print(f"\nNumber of seed features:   {len(seed_features)}")
print(f"Number of final features:  {len(df_final)}")
print("\nDone.")

### Creating subsets of features

In [ ]:
#!/usr/bin/env python3
"""
MINIMAL SUBSET SEARCH FROM PRE-RANKED FEATURES
==============================================

Purpose
-------
Run exhaustive subset search directly from feature_ranking_pre.csv,
without any spike analysis or curve-based candidate selection.

Inputs
------
- feature_ranking_pre.csv
- EOF_latdepth_{var}_{lon}.nc
- AMOC_{MODEL}.nc

Outputs
-------
- best_subsets_by_k.csv
- all_subsets_scored.csv   (optional)
- summary_best_subsets.txt

Notes
-----
- Candidate features are taken directly from the top rows of feature_ranking_pre.csv
- Lag filtering can still be applied (for precursor-focused analyses)
- Selection still be based on TRAIN or TEST R²
"""

import os
import glob
import itertools
import numpy as np
import pandas as pd
import xarray as xr

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# ============================================================
# USER SETTINGS
# ============================================================
MODEL  = "EC-Earth3"
TARGET = "AMOC_26N_ensmean"

# Choose one:
MODE = "ensmean"
member_id = None

# Example for member mode:
# MODE = "member"
# member_id = "r1i1p1f1"

# Choose AMOC variable:
#   "normal" -> use TARGET as-is
#   "smooth" -> map to AMOC_26N_smooth / AMOC_45N_smooth
AMOC_VARIANT = "normal"   # "normal" or "smooth"

EOF_DIR   = f"/data/projects/nckf/frekle/EOF_results/{MODEL}/latdepth_sections/Train_period_85pct/"
AMOC_FILE = f"/data/users/frekle/AMOC_analysis/AMOC_{MODEL}.nc"
RANK_CSV  = f"/data/users/frekle/Final_figures/{MODEL}/{TARGET}/Feature_selection/feature_ranking_pre.csv"

# Candidate selection from ranking
N_CANDIDATES = 10   # use top N ranked features directly

# Subset selection criterion
SELECT_ON   = "train"   # "train" or "test"
TIEBREAK_ON = "test"    # "train" or "test"

# Precursor lag constraint
MIN_LAG = 4
MAX_LAG = None   # e.g. 20, or None

# Window choice:
#   "maxlag" -> evaluation starts at max lag among candidate features
#   "minlag" -> evaluation starts at MIN_LAG
WINDOW_LAG_POLICY = "minlag"   # "maxlag" or "minlag"

OUTDIR = (
    f"/data/users/frekle/Final_figures/{MODEL}/{TARGET}/Feature_selection/"
    f"Not_detrended/Selected_on_{SELECT_ON}/amoc_variant_{AMOC_VARIANT}/"
    f"lag_policy_{MIN_LAG}minlag_{MAX_LAG if MAX_LAG is not None else 'None'}max/"
)
os.makedirs(OUTDIR, exist_ok=True)

# PC setup
N_MODES = 10

# Ridge hyperparameters
ALPHA = 1.0

# Detrending (TRAIN-fit)
DETREND_Y = False
DETREND_X = False

# Subset search settings
MAX_FEATURES_TO_USE = 3
SAVE_ALL_SUBSETS = True


# ============================================================
# Helpers
# ============================================================
def extract_years(time_coord):
    try:
        return xr.DataArray(time_coord).dt.year.values.astype(int)
    except Exception:
        t = np.asarray(time_coord)
        return np.array([int(str(x)[:4]) for x in t], dtype=int)


def load_pc_latdepth(var, lon_tag, n_modes, eof_dir, mode="ensmean", member_id=None):
    f = os.path.join(eof_dir, f"EOF_latdepth_{var}_{lon_tag}.nc")
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing EOF file: {f}")

    ds = xr.open_dataset(f)
    if "PC" not in ds:
        ds.close()
        raise KeyError(f"'PC' not found in {f}")

    PC = ds["PC"].isel(mode=slice(0, int(n_modes)))

    if "member" in PC.dims:
        if mode == "ensmean":
            PC = PC.mean("member")
        elif mode == "member":
            if member_id is None:
                ds.close()
                raise ValueError("member_id must be provided when mode='member'")
            if "member" in PC.coords and member_id in PC["member"].values:
                PC = PC.sel(member=member_id)
            else:
                PC = PC.isel(member=int(member_id))
        else:
            ds.close()
            raise ValueError("mode must be 'ensmean' or 'member'")

    PC = PC.transpose("time", "mode")
    years = extract_years(PC["time"])
    PC = PC.assign_coords(year=("time", years)).swap_dims({"time": "year"}).drop_vars("time")

    ds.close()
    return PC.astype(float)


def _infer_amoc_lat_from_target(target_name: str):
    if "26N" in target_name:
        return "26N"
    if "45N" in target_name:
        return "45N"
    raise ValueError(f"Could not infer latitude tag (26N/45N) from TARGET='{target_name}'")


def resolve_amoc_variable(target: str, amoc_variant: str):
    amoc_variant = str(amoc_variant).strip().lower()
    if amoc_variant == "normal":
        return target
    if amoc_variant == "smooth":
        lat = _infer_amoc_lat_from_target(target)
        return f"AMOC_{lat}_smooth"
    raise ValueError("AMOC_VARIANT must be 'normal' or 'smooth'")


def load_amoc(target, amoc_file, mode="ensmean", member_id=None, amoc_variant="normal"):
    varname = resolve_amoc_variable(target, amoc_variant)

    ds = xr.open_dataset(amoc_file)
    if varname not in ds:
        ds.close()
        raise KeyError(f"AMOC var '{varname}' not found in {amoc_file}. Available: {list(ds.data_vars)}")

    y = ds[varname].squeeze()

    if "year" in y.dims:
        am = y
    else:
        years = extract_years(y["time"])
        am = y.assign_coords(year=("time", years)).swap_dims({"time": "year"}).drop_vars("time")

    am = am.astype(float)

    if "member" in am.dims:
        if mode == "ensmean":
            am = am.mean("member")
        elif mode == "member":
            if member_id is None:
                ds.close()
                raise ValueError("member_id must be provided when mode='member'")
            if "member" in am.coords and member_id in am["member"].values:
                am = am.sel(member=member_id)
            else:
                am = am.isel(member=int(member_id))
        else:
            ds.close()
            raise ValueError("mode must be 'ensmean' or 'member'")

    ds.close()
    return am.squeeze()


def infer_train_end_year_from_any_eof(eof_dir, prefer=("so", "W30p0")):
    var, lon = prefer
    f = os.path.join(eof_dir, f"EOF_latdepth_{var}_{lon}.nc")
    if not os.path.exists(f):
        hits = sorted(glob.glob(os.path.join(eof_dir, "EOF_latdepth_*_*.nc")))
        if not hits:
            raise FileNotFoundError(f"No EOF files found in {eof_dir}")
        f = hits[0]

    ds = xr.open_dataset(f)

    if "TRAIN_END" in ds.attrs:
        y = int(str(ds.attrs["TRAIN_END"])[:4])
        ds.close()
        return y

    if "train_mask" in ds:
        tm = ds["train_mask"].values.astype(bool)
        if tm.any():
            t_last = ds["time"].values[np.where(tm)[0][-1]]
            ds.close()
            return int(str(np.datetime64(t_last))[:4])

    ds.close()
    raise RuntimeError(f"Could not infer TRAIN_END year from {f}")


def fit_linear_trend(train_years, train_series):
    x = np.asarray(train_years, float)
    y = np.asarray(train_series, float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]

    if len(x) < 2:
        return 0.0, float(np.nanmean(y))

    A = np.vstack([x, np.ones_like(x)]).T
    a, b = np.linalg.lstsq(A, y, rcond=None)[0]
    return float(a), float(b)


def detrend_with_train_fit(all_years, all_series, train_mask):
    all_years = np.asarray(all_years, float)
    all_series = np.asarray(all_series, float)
    a, b = fit_linear_trend(all_years[train_mask], all_series[train_mask])
    trend = a * all_years + b
    return all_series - trend, (a, b)


def pearson_corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 2:
        return np.nan
    a = a[m]
    b = b[m]
    sa = a.std()
    sb = b.std()
    if sa == 0 or sb == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def finite_rows_mask(Y, X):
    Y = np.asarray(Y, float)
    X = np.asarray(X, float)
    m = np.isfinite(Y)
    if X.ndim == 1:
        m = m & np.isfinite(X)
    else:
        m = m & np.all(np.isfinite(X), axis=1)
    return m


def safe_r2(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    if len(y_true) < 2:
        return np.nan
    return float(r2_score(y_true, y_pred))


def choose_better(curr, best, select_on="train", tiebreak_on="test"):
    if select_on not in ("train", "test"):
        raise ValueError("SELECT_ON must be 'train' or 'test'")
    if tiebreak_on not in ("train", "test"):
        raise ValueError("TIEBREAK_ON must be 'train' or 'test'")

    key_main = "r2_train" if select_on == "train" else "r2_test"
    key_tie  = "r2_train" if tiebreak_on == "train" else "r2_test"

    if curr[key_main] > best[key_main]:
        return True
    if np.isclose(curr[key_main], best[key_main]) and curr[key_tie] > best[key_tie]:
        return True
    return False


def load_ranked_features(csv_path):
    """
    Accepts either lon_tag or lon.
    Expects columns: var, (lon_tag|lon), mode, lag
    mode is assumed 1-based in file and converted to 0-based internally.
    """
    df = pd.read_csv(csv_path)

    if "lon" not in df.columns and "lon_tag" in df.columns:
        df = df.rename(columns={"lon_tag": "lon"})
    if "lon" not in df.columns:
        raise KeyError(f"Missing lon/lon_tag in {csv_path}. Columns: {list(df.columns)}")

    for c in ["var", "lon", "mode", "lag"]:
        if c not in df.columns:
            raise KeyError(f"Missing column '{c}' in {csv_path}. Columns: {list(df.columns)}")

    df["var"] = df["var"].astype(str).str.strip()
    df["lon"] = df["lon"].astype(str).str.strip()
    df["mode0"] = df["mode"].astype(int) - 1
    df["lag"] = df["lag"].astype(int)

    feats = [(r["var"], r["lon"], int(r["mode0"]), int(r["lag"])) for _, r in df.iterrows()]
    return feats, df


def filter_feats_by_lag(feats, min_lag=0, max_lag=None):
    out = []
    for (v, lon, m0, lag) in feats:
        if lag < int(min_lag):
            continue
        if max_lag is not None and lag > int(max_lag):
            continue
        out.append((v, lon, m0, lag))
    return out


# ============================================================
# 1) Build candidate feature list directly from ranking
# ============================================================
feats_ranked, _ = load_ranked_features(RANK_CSV)

cand_feats = feats_ranked[:N_CANDIDATES]
cand_feats = filter_feats_by_lag(cand_feats, min_lag=MIN_LAG, max_lag=MAX_LAG)

# remove duplicates while preserving order
seen = set()
cand_feats_unique = []
for ft in cand_feats:
    if ft not in seen:
        cand_feats_unique.append(ft)
        seen.add(ft)
cand_feats = cand_feats_unique

M = len(cand_feats)
if M == 0:
    raise RuntimeError(
        f"No candidate features left after lag filtering: MIN_LAG={MIN_LAG}, MAX_LAG={MAX_LAG}"
    )

print("\n==============================")
print("RANKING-BASED CANDIDATE FEATURES")
print("==============================")
print(f"Top ranked features used: {N_CANDIDATES}")
print(f"Candidate features after lag filter: {M}")
print(f"MIN_LAG filter: >= {MIN_LAG}" + (f", <= {MAX_LAG}" if MAX_LAG is not None else ""))
for i, (v, lon, m0, lag) in enumerate(cand_feats, 1):
    print(f"  C{i}: {v} {lon} EOF{m0+1} lag{lag}")

KMAX = min(MAX_FEATURES_TO_USE, M)


# ============================================================
# 2) Load AMOC + PCs needed
# ============================================================
TRAIN_END_YEAR = infer_train_end_year_from_any_eof(EOF_DIR, prefer=("so", "W30p0"))
print("\n✅ TRAIN_END_YEAR inferred from EOF files:", TRAIN_END_YEAR)

amoc_var_used = resolve_amoc_variable(TARGET, AMOC_VARIANT)
print(f"✅ AMOC variable used: {amoc_var_used}   (AMOC_VARIANT={AMOC_VARIANT})")

amoc = load_amoc(TARGET, AMOC_FILE, mode=MODE, member_id=member_id, amoc_variant=AMOC_VARIANT)

needed_pairs = sorted(set((v, lon) for (v, lon, _, _) in cand_feats))
pc_dict = {
    (v, lon): load_pc_latdepth(v, lon, N_MODES, EOF_DIR, mode=MODE, member_id=member_id)
    for (v, lon) in needed_pairs
}

common_years = amoc["year"].values.astype(int)
for da in pc_dict.values():
    common_years = np.intersect1d(common_years, da["year"].values.astype(int))

years = np.asarray(common_years, int)
years.sort()

if len(years) == 0:
    raise RuntimeError("No overlapping years found between AMOC and PCs.")

y_amoc = amoc.sel(year=years).values.astype(float)
pc_np = {k: v.sel(year=years).values.astype(float) for k, v in pc_dict.items()}


# ============================================================
# 3) Fix evaluation window for fair comparison
# ============================================================
GLOBAL_MAXLAG = max(lag for *_, lag in cand_feats)

if WINDOW_LAG_POLICY == "maxlag":
    START_LAG = GLOBAL_MAXLAG
elif WINDOW_LAG_POLICY == "minlag":
    START_LAG = int(MIN_LAG)
else:
    raise ValueError("WINDOW_LAG_POLICY must be 'maxlag' or 'minlag'")

used_global = np.arange(START_LAG, len(years), dtype=int)
years_used = years[used_global]
Y_all = y_amoc[used_global]

idx_tr = np.where(years_used <= TRAIN_END_YEAR)[0]
idx_te = np.where(years_used > TRAIN_END_YEAR)[0]

if len(idx_tr) == 0:
    raise RuntimeError("Train period is empty after windowing.")
if len(idx_te) == 0:
    raise RuntimeError("Test period is empty after windowing.")

print("\nSplit on years_used:")
print("GLOBAL_MAXLAG (of candidates):", GLOBAL_MAXLAG)
print("START_LAG (window starts):", START_LAG, f"(policy={WINDOW_LAG_POLICY})")
print("Train:", years_used[idx_tr[0]], "–", years_used[idx_tr[-1]], "n=", len(idx_tr))
print("Test :", years_used[idx_te[0]], "–", years_used[idx_te[-1]], "n=", len(idx_te))

train_mask_used = np.zeros(len(years_used), dtype=bool)
train_mask_used[idx_tr] = True

if DETREND_Y:
    Y_dt, (ay, by) = detrend_with_train_fit(years_used, Y_all, train_mask_used)
    print(f"✅ Detrended Y using TRAIN fit: y_trend = {ay:.4e}*year + {by:.4e}")
else:
    Y_dt = Y_all.copy()

X_cand = np.empty((len(used_global), M), dtype=float)
for j, (var, lon, m0, lag) in enumerate(cand_feats):
    X_cand[:, j] = pc_np[(var, lon)][used_global - lag, m0]

if DETREND_X:
    X_cand_dt = X_cand.copy()
    for j in range(M):
        X_cand_dt[:, j], _ = detrend_with_train_fit(years_used, X_cand[:, j], train_mask_used)
else:
    X_cand_dt = X_cand


# ============================================================
# 4) Exhaustive subset search
# ============================================================
all_rows = []
best_by_k = []

print("\n==============================")
print("SUBSET SEARCH")
print("==============================")
print(f"Selecting BEST subsets by: {SELECT_ON.upper()} R² (tie-break: {TIEBREAK_ON.upper()} R²)")
print(f"Target AMOC: {amoc_var_used}")
print(f"ALPHA={ALPHA}  DETREND_Y={DETREND_Y}  DETREND_X={DETREND_X}")
print(f"KMAX={KMAX}  Candidates={M}\n")

for k in range(1, KMAX + 1):
    best = {
        "k": k,
        "r2_train": -np.inf,
        "corr_train": np.nan,
        "r2_test": -np.inf,
        "corr_test": np.nan,
        "subset_idx": None,
    }

    for subset_idx in itertools.combinations(range(M), k):
        subset_idx = tuple(subset_idx)
        Xk = X_cand_dt[:, subset_idx]

        mdl = make_pipeline(StandardScaler(), Ridge(alpha=float(ALPHA)))

        Ytr = Y_dt[idx_tr]
        Xtr = Xk[idx_tr]
        m_tr = finite_rows_mask(Ytr, Xtr)

        Yte = Y_dt[idx_te]
        Xte = Xk[idx_te]
        m_te = finite_rows_mask(Yte, Xte)

        if m_tr.sum() < 3:
            continue
        if m_te.sum() < 2:
            continue

        mdl.fit(Xtr[m_tr], Ytr[m_tr])

        pred_tr = mdl.predict(Xtr[m_tr])
        r2_tr = safe_r2(Ytr[m_tr], pred_tr)
        c_tr = pearson_corr(Ytr[m_tr], pred_tr)

        pred_te = mdl.predict(Xte[m_te])
        r2_te = safe_r2(Yte[m_te], pred_te)
        c_te = pearson_corr(Yte[m_te], pred_te)

        if not np.isfinite(r2_tr) or not np.isfinite(r2_te):
            continue

        curr = {
            "r2_train": r2_tr,
            "corr_train": c_tr,
            "r2_test": r2_te,
            "corr_test": c_te,
            "subset_idx": subset_idx,
        }

        if SAVE_ALL_SUBSETS:
            subset_feats = [cand_feats[i] for i in subset_idx]
            subset_str = " | ".join(
                [f"{v} {lon} EOF{m0+1} lag{lag}" for (v, lon, m0, lag) in subset_feats]
            )
            all_rows.append({
                "k": k,
                "r2_train": r2_tr,
                "corr_train": c_tr,
                "r2_test": r2_te,
                "corr_test": c_te,
                "subset_idx": ",".join(map(str, subset_idx)),
                "subset_features": subset_str,
                "selected_by": SELECT_ON,
                "amoc_variant": AMOC_VARIANT,
                "amoc_var_used": amoc_var_used,
            })

        if best["subset_idx"] is None or choose_better(curr, best, select_on=SELECT_ON, tiebreak_on=TIEBREAK_ON):
            best["r2_train"] = r2_tr
            best["corr_train"] = c_tr
            best["r2_test"] = r2_te
            best["corr_test"] = c_te
            best["subset_idx"] = subset_idx

    if best["subset_idx"] is None:
        print(f"No valid subset found for k={k}")
        continue

    subset_feats = [cand_feats[i] for i in best["subset_idx"]]
    subset_str = " | ".join([f"{v} {lon} EOF{m0+1} lag{lag}" for (v, lon, m0, lag) in subset_feats])

    best_by_k.append({
        "k": k,
        "r2_train": best["r2_train"],
        "corr_train": best["corr_train"],
        "r2_test": best["r2_test"],
        "corr_test": best["corr_test"],
        "subset_idx": ",".join(map(str, best["subset_idx"])),
        "subset_features": subset_str,
        "selected_by": SELECT_ON,
        "amoc_variant": AMOC_VARIANT,
        "amoc_var_used": amoc_var_used,
    })

    print(
        f"BEST k={k} (selected on {SELECT_ON}): "
        f"Train R²={best['r2_train']:.4f} corr={best['corr_train']:.4f} | "
        f"Test R²={best['r2_test']:.4f} corr={best['corr_test']:.4f}"
    )
    for (v, lon, m0, lag) in subset_feats:
        print(f"  - {v:6s} {lon:6s} EOF{m0+1:<2d} lag{lag}")
    print("")


# ============================================================
# 5) Save results
# ============================================================
best_df = pd.DataFrame(best_by_k)
best_csv = os.path.join(OUTDIR, "best_subsets_by_k.csv")
best_df.to_csv(best_csv, index=False)
print("\n✅ Saved best subsets by k:", best_csv)

if SAVE_ALL_SUBSETS:
    sort_main = "r2_train" if SELECT_ON == "train" else "r2_test"
    sort_tie  = "r2_train" if TIEBREAK_ON == "train" else "r2_test"

    all_df = (
        pd.DataFrame(all_rows)
        .sort_values(["k", sort_main, sort_tie], ascending=[True, False, False])
        .reset_index(drop=True)
    )

    all_csv = os.path.join(OUTDIR, "all_subsets_scored.csv")
    all_df.to_csv(all_csv, index=False)
    print("✅ Saved all subset scores:", all_csv)

summary_txt = os.path.join(OUTDIR, "summary_best_subsets.txt")
with open(summary_txt, "w") as f:
    f.write(f"MODEL={MODEL}\n")
    f.write(f"TARGET={TARGET}\n")
    f.write(f"MODE={MODE}\n")
    f.write(f"AMOC_VARIANT={AMOC_VARIANT}\n")
    f.write(f"AMOC_VAR_USED={amoc_var_used}\n")
    f.write(f"TRAIN_END_YEAR={TRAIN_END_YEAR}\n")
    f.write(f"GLOBAL_MAXLAG={GLOBAL_MAXLAG}\n")
    f.write(f"ALPHA={ALPHA}\n")
    f.write(f"DETREND_Y={DETREND_Y}\n")
    f.write(f"DETREND_X={DETREND_X}\n")
    f.write(f"RANK_CSV={RANK_CSV}\n")
    f.write(f"N_CANDIDATES={N_CANDIDATES}\n")
    f.write(f"SELECT_ON={SELECT_ON}\n")
    f.write(f"TIEBREAK_ON={TIEBREAK_ON}\n")
    f.write(f"MIN_LAG={MIN_LAG}\n")
    f.write(f"MAX_LAG={MAX_LAG}\n")
    f.write(f"WINDOW_LAG_POLICY={WINDOW_LAG_POLICY}\n")

    f.write("\nBest subsets by k:\n")
    for row in best_by_k:
        f.write(
            f"\nK={row['k']}  "
            f"R2_train={row['r2_train']:.6f}  corr_train={row['corr_train']:.6f}  "
            f"R2_test={row['r2_test']:.6f}  corr_test={row['corr_test']:.6f}\n"
        )
        f.write(row["subset_features"] + "\n")

print("✅ Saved summary:", summary_txt)
print("\nDONE.")

### Figures

In [ ]:
#!/usr/bin/env python3
"""
FINAL FIGURES
================================================================

This script creates the main figures for the selected feature subsets.

It produces:
  (1) Reconstruction figures for k=1..K_MAX
  (2) EOF(lat-depth) + PC mode panels for unique selected mode-sets
  (3) Two-panel EWS figures for unique selected mode-sets:
        - top: rolling lag-1 autocorrelation (AR1)
        - bottom: rolling variance
  (4) Contribution plot showing cumulative TEST R² by model size k

Inputs expected:
  - subset search output: best_subsets_by_k.csv

Outputs:
  - 01_reconstruction_k1..kKMAX.(png/pdf)
  - 01_reconstruction_ALL_k1toK.(png/pdf)
  - 01_reconstruction_PAPER_READY_...(png/pdf)
  - 02_EOFmap_and_PC_...(png/pdf)
  - 03_EWS_twopanel_...(png/pdf)
  - 05_r2_contribution_waterfall_by_k_stacked.(png/pdf)
  - reconstruction_summary_by_k.csv
  - 05_r2_contribution_by_k.csv

Notes:
  - Split year is inferred from EOF file TRAIN_END or train_mask.
  - Detrending uses TRAIN-fit only (no leakage) if enabled.
  - Reconstruction uses fixed alpha ALPHA_FIXED.
  - Reconstruction scoring uses TEST window (post TRAIN_END_YEAR).
  - No spike-based figures are included.
"""

import os
import re
import glob
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

from matplotlib.ticker import MultipleLocator
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# ============================================================
# USER SETTINGS
# ============================================================
MODEL  = "CESM2"   # "EC-Earth3", "IPSL-CM6A-LR", "CESM2", "MPI-ESM1-2-LR"
TARGET = "AMOC_45N_ensmean"
MODE   = "ensmean"      # "ensmean" or "member"
member_id = None        # if MODE="member": label or integer index

EOF_DIR   = f"/data/projects/nckf/frekle/EOF_results/{MODEL}/latdepth_sections/Train_period_85pct/"
AMOC_FILE = f"/data/users/frekle/AMOC_analysis/AMOC_{MODEL}.nc"

# Choose which AMOC series to PLOT/RECONSTRUCT as "truth"
#   "normal" -> uses TARGET as-is
#   "smooth" -> uses AMOC_26N_smooth or AMOC_45N_smooth (based on TARGET latitude)
AMOC_VARIANT = "normal"   # "normal" or "smooth"

# Input produced by your pipeline
BEST_CSV = (
    f"/data/users/frekle/Final_figures/{MODEL}/{TARGET}/Feature_selection/"
    f"Not_detrended/Selected_on_train/amoc_variant_{AMOC_VARIANT}/"
    f"lag_policy_4minlag_Nonemax/best_subsets_by_k.csv"
)

# Output
OUTDIR = (
    f"/data/users/frekle/Final_figures/{MODEL}/{TARGET}/Best_test_R2/"
    f"Not_detrended/Selected_on_train/amoc_variant_{AMOC_VARIANT}/minlag_4/"
)
os.makedirs(OUTDIR, exist_ok=True)

# Figure / feature settings
K_MAX = 5
MAX_MODE_PANELS = 5
N_MODES = 10

# Reconstruction preprocessing
DETREND_Y = False
DETREND_X = False

# Alpha (fixed; no tuning)
ALPHA_FIXED = 1.0

# AR1
AR1_WINDOW_YEARS = 30
AR1_MIN_VALID = 10

# Variance
VAR_WINDOW_YEARS = 30
VAR_MIN_VALID = 10
VAR_DDOF = 1
VAR_STANDARDIZE_FOR_PLOTTING = True

# Aesthetic controls
PC_SPAGHETTI_ALPHA = 0.25
PC_SPAGHETTI_LW = 0.6
PC_MEAN_LW = 1.8
LINEWIDTH_PC = 1.8
LINEWIDTH_AMOC = 2.2


# ============================================================
# HELPERS
# ============================================================
def savefig(outbase, dpi=220):
    plt.savefig(outbase + ".png", dpi=dpi, bbox_inches="tight")
    plt.savefig(outbase + ".pdf", dpi=dpi, bbox_inches="tight")


def extract_years(time_coord):
    try:
        return xr.DataArray(time_coord).dt.year.values.astype(int)
    except Exception:
        t = np.asarray(time_coord)
        return np.array([int(str(x)[:4]) for x in t], dtype=int)


def format_year_axis(ax, years_arr, step=10, pad=5):
    years_arr = np.asarray(years_arr, int)
    ax.set_xlim(int(years_arr.min()) - pad, int(years_arr.max()) + pad)
    ax.xaxis.set_major_locator(MultipleLocator(step))


def infer_train_end_year_from_any_eof(eof_dir, prefer=("so", "W30p0")):
    var, lon = prefer
    f = os.path.join(eof_dir, f"EOF_latdepth_{var}_{lon}.nc")
    if not os.path.exists(f):
        hits = sorted(glob.glob(os.path.join(eof_dir, "EOF_latdepth_*_*.nc")))
        if not hits:
            raise FileNotFoundError(f"No EOF files found in {eof_dir}")
        f = hits[0]

    ds = xr.open_dataset(f)
    if "TRAIN_END" in ds.attrs:
        y = int(str(ds.attrs["TRAIN_END"])[:4])
        ds.close()
        return y

    if "train_mask" in ds:
        tm = ds["train_mask"].values.astype(bool)
        if tm.any():
            t_last = ds["time"].values[np.where(tm)[0][-1]]
            ds.close()
            return int(str(np.datetime64(t_last))[:4])

    ds.close()
    raise RuntimeError(f"Could not infer TRAIN_END year from {f}")


def _infer_amoc_lat_from_target(target_name: str):
    if "26N" in target_name:
        return "26N"
    if "45N" in target_name:
        return "45N"
    raise ValueError(f"Could not infer latitude tag (26N/45N) from TARGET='{target_name}'")


def resolve_amoc_variable(target: str, amoc_variant: str):
    amoc_variant = str(amoc_variant).strip().lower()
    if amoc_variant == "normal":
        return target
    if amoc_variant == "smooth":
        lat = _infer_amoc_lat_from_target(target)
        return f"AMOC_{lat}_smooth"
    raise ValueError("AMOC_VARIANT must be 'normal' or 'smooth'")


def load_amoc(target, amoc_file, mode="ensmean", member_id=None, amoc_variant="normal"):
    varname = resolve_amoc_variable(target, amoc_variant)

    ds = xr.open_dataset(amoc_file)
    if varname not in ds.data_vars:
        ds.close()
        raise KeyError(f"AMOC var '{varname}' not found in {amoc_file}. Available: {list(ds.data_vars)}")

    y = ds[varname].squeeze()

    if "year" in y.dims:
        yy = y
        if "time" in yy.coords:
            yy = yy.drop_vars("time")
    else:
        years = extract_years(y["time"])
        yy = y.assign_coords(year=("time", years)).swap_dims({"time": "year"}).drop_vars("time")

    yy = yy.astype(float)

    if "member" in yy.dims:
        if mode == "ensmean":
            yy = yy.mean("member")
        elif mode == "member":
            if member_id is None:
                ds.close()
                raise ValueError("member_id must be provided when mode='member'")
            if "member" in yy.coords and member_id in yy["member"].values:
                yy = yy.sel(member=member_id)
            else:
                yy = yy.isel(member=int(member_id))
        else:
            ds.close()
            raise ValueError("mode must be 'ensmean' or 'member'")

    ds.close()
    return yy.squeeze()


def load_pc_latdepth(var, lon_tag, n_modes, eof_dir, mode="ensmean", member_id=None):
    f = os.path.join(eof_dir, f"EOF_latdepth_{var}_{lon_tag}.nc")
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing EOF file: {f}")

    ds = xr.open_dataset(f)
    PC = ds["PC"].isel(mode=slice(0, int(n_modes)))

    if "member" in PC.dims:
        if mode == "ensmean":
            PC = PC.mean("member")
        else:
            if member_id is None:
                ds.close()
                raise ValueError("member_id must be provided when mode='member'")
            if "member" in PC.coords and member_id in PC["member"].values:
                PC = PC.sel(member=member_id)
            else:
                PC = PC.isel(member=int(member_id))

    PC = PC.transpose("time", "mode")
    years = extract_years(PC["time"])
    PC = PC.assign_coords(year=("time", years)).swap_dims({"time": "year"}).drop_vars("time")
    ds.close()
    return PC.astype(float)


def load_pc_latdepth_with_members(var, lon_tag, n_modes, eof_dir):
    f = os.path.join(eof_dir, f"EOF_latdepth_{var}_{lon_tag}.nc")
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing EOF file: {f}")

    ds = xr.open_dataset(f)
    PC = ds["PC"].isel(mode=slice(0, int(n_modes)))
    years = extract_years(PC["time"])
    PC = PC.assign_coords(year=("time", years)).swap_dims({"time": "year"}).drop_vars("time")
    ds.close()
    return PC.astype(float)


def load_eof_map_latdepth(var, lon_tag, eof_dir, mode_index_1based):
    f = os.path.join(eof_dir, f"EOF_latdepth_{var}_{lon_tag}.nc")
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing EOF file: {f}")

    ds = xr.open_dataset(f)

    eof_var_candidates = ["EOF", "eof", "patterns", "EOFs"]
    eof_name = next((c for c in eof_var_candidates if c in ds.data_vars), None)
    if eof_name is None:
        raise KeyError(f"Could not find EOF variable in {f}. Available: {list(ds.data_vars)}")

    EOF = ds[eof_name]
    m0 = int(mode_index_1based) - 1
    EOFm = EOF.isel(mode=m0)

    depth_dim_candidates = ["depth", "lev", "z", "olevel"]
    depth_dim = next((d for d in depth_dim_candidates if d in EOFm.dims), None)
    if depth_dim is None:
        raise KeyError(f"Could not detect depth dim in EOF. dims={EOFm.dims}")

    lat_dim_candidates = ["lat", "latitude", "lat_index", "y", "j"]
    lat_dim = next((d for d in lat_dim_candidates if d in EOFm.dims), None)
    if lat_dim is None:
        others = [d for d in EOFm.dims if d != depth_dim]
        if len(others) != 1:
            raise KeyError(f"Could not detect lat-like dim. dims={EOFm.dims}")
        lat_dim = others[0]

    EOFm = EOFm.transpose(depth_dim, lat_dim)

    depth = EOFm[depth_dim].values.astype(float)

    if "lat" in EOFm.coords:
        lat = EOFm["lat"].values.astype(float)
    elif lat_dim in EOFm.coords:
        lat = EOFm[lat_dim].values.astype(float)
    else:
        lat = np.arange(EOFm.sizes[lat_dim], dtype=float)

    Z = EOFm.values.astype(float)
    ds.close()

    if Z.shape[-1] != len(lat):
        raise ValueError(f"Lat length mismatch: Z.shape={Z.shape}, len(lat)={len(lat)}")

    return Z, depth, lat


def fit_linear_trend(train_years, train_series):
    x = np.asarray(train_years, float)
    y = np.asarray(train_series, float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]
    if len(x) < 2:
        return 0.0, float(np.nanmean(y))
    A = np.vstack([x, np.ones_like(x)]).T
    a, b = np.linalg.lstsq(A, y, rcond=None)[0]
    return float(a), float(b)


def detrend_with_train_fit(all_years, all_series, train_mask):
    all_years = np.asarray(all_years, float)
    all_series = np.asarray(all_series, float)
    a, b = fit_linear_trend(all_years[train_mask], all_series[train_mask])
    return all_series - (a * all_years + b), (a, b)


def standardize_with_train_fit(all_series, train_mask):
    x = np.asarray(all_series, float)
    train_x = x[train_mask]
    train_x = train_x[np.isfinite(train_x)]

    if len(train_x) < 2:
        mu = float(np.nanmean(x))
        sigma = float(np.nanstd(x))
    else:
        mu = float(np.nanmean(train_x))
        sigma = float(np.nanstd(train_x, ddof=1))

    if not np.isfinite(sigma) or sigma == 0:
        sigma = 1.0

    z = (x - mu) / sigma
    return z, (mu, sigma)


def pearson_corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 3:
        return np.nan
    a = a[m]
    b = b[m]
    if a.std() == 0 or b.std() == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def rolling_ar1(x, years, window=30, min_valid=10):
    x = np.asarray(x, float)
    years = np.asarray(years, int)
    n = len(x)

    if n < window:
        window = max(5, n // 2)

    ar1_vals = np.full(n, np.nan, float)
    half = window // 2

    for i in range(n):
        lo = max(0, i - half)
        hi = min(n, i + half + 1)

        seg = x[lo:hi]
        m = np.isfinite(seg)
        seg = seg[m]

        if len(seg) < max(min_valid, 3):
            continue

        a = seg[1:]
        b = seg[:-1]
        if len(a) < 2:
            continue

        ar1_vals[i] = pearson_corr(a, b)

    return years, ar1_vals


def rolling_variance(x, years, window=30, min_valid=10, ddof=1):
    x = np.asarray(x, float)
    years = np.asarray(years, int)
    n = len(x)

    if n < window:
        window = max(5, n // 2)

    var_vals = np.full(n, np.nan, float)
    half = window // 2

    for i in range(n):
        lo = max(0, i - half)
        hi = min(n, i + half + 1)

        seg = x[lo:hi]
        seg = seg[np.isfinite(seg)]

        if len(seg) < max(min_valid, 2):
            continue
        if len(seg) <= ddof:
            continue

        var_vals[i] = float(np.var(seg, ddof=ddof))

    return years, var_vals


def parse_features_string(s):
    feats = []
    for part in str(s).split("|"):
        part = part.strip()
        m = re.search(r"(\w+)\s+(W\d+p\d+)\s+EOF(\d+)\s+lag(\d+)", part)
        if not m:
            continue
        var, lon, eofk, lag = m.group(1), m.group(2), int(m.group(3)), int(m.group(4))
        feats.append((var, lon, eofk - 1, lag))

    out, seen = [], set()
    for ft in feats:
        if ft not in seen:
            out.append(ft)
            seen.add(ft)
    return out


def finite_rows_mask(Y, X):
    Y = np.asarray(Y, float)
    X = np.asarray(X, float)
    m = np.isfinite(Y)
    if X.ndim == 1:
        m = m & np.isfinite(X)
    else:
        m = m & np.all(np.isfinite(X), axis=1)
    return m


def safe_r2(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    if len(y_true) < 2:
        return np.nan
    return float(r2_score(y_true, y_pred))


# ============================================================
# LOAD GLOBAL STUFF
# ============================================================
TRAIN_END_YEAR = infer_train_end_year_from_any_eof(EOF_DIR, prefer=("so", "W30p0"))
print("✅ TRAIN_END_YEAR:", TRAIN_END_YEAR)

amoc_var_used = resolve_amoc_variable(TARGET, AMOC_VARIANT)
print(f"✅ AMOC truth used in figures: {amoc_var_used} (AMOC_VARIANT={AMOC_VARIANT})")

amoc = load_amoc(TARGET, AMOC_FILE, mode=MODE, member_id=member_id, amoc_variant=AMOC_VARIANT)

df_best = pd.read_csv(BEST_CSV)
df_best = df_best.sort_values("k")
available_ks = sorted(df_best["k"].unique().tolist())
K_MAX_USE = min(K_MAX, int(max(available_ks)) if available_ks else 0)
if K_MAX_USE < 1:
    raise RuntimeError(f"No k rows found in {BEST_CSV}")

print("✅ Will generate k=1..", K_MAX_USE)


# ============================================================
# FUNCTION: build XY for a given feature subset
# ============================================================
def build_XY_for_feats(feats, years, y_amoc, pc_np, global_maxlag=None):
    if global_maxlag is None:
        maxlag = int(max(lag for *_, lag in feats))
    else:
        maxlag = int(global_maxlag)

    used = np.arange(maxlag, len(years), dtype=int)
    years_used = years[used]
    Y = y_amoc[used].astype(float)

    X = np.empty((len(used), len(feats)), float)
    for j, (v, lon, m0, lag) in enumerate(feats):
        X[:, j] = pc_np[(v, lon)][used - int(lag), int(m0)]

    idx_tr = np.where(years_used <= TRAIN_END_YEAR)[0]
    idx_te = np.where(years_used > TRAIN_END_YEAR)[0]

    if len(idx_tr) < 5 or len(idx_te) < 5:
        raise RuntimeError(
            f"Too few samples after split: train={len(idx_tr)} test={len(idx_te)} "
            f"(years_used={years_used[0]}–{years_used[-1]}, TRAIN_END_YEAR={TRAIN_END_YEAR})"
        )

    train_mask_used = np.zeros(len(years_used), dtype=bool)
    train_mask_used[idx_tr] = True

    if DETREND_Y:
        Y_dt, (ay, by) = detrend_with_train_fit(years_used, Y, train_mask_used)
    else:
        Y_dt = Y.copy()
        ay, by = 0.0, 0.0

    if DETREND_X:
        X_dt = X.copy()
        for j in range(X_dt.shape[1]):
            X_dt[:, j], _ = detrend_with_train_fit(years_used, X[:, j], train_mask_used)
    else:
        X_dt = X.copy()

    return years_used, X_dt, Y_dt, idx_tr, idx_te, (ay, by), used


# ============================================================
# PRELOAD PCs needed for all k
# ============================================================
all_feats_union = []
for k in range(1, K_MAX_USE + 1):
    row = df_best.loc[df_best["k"] == k]
    if len(row) != 1:
        continue
    feats = parse_features_string(row.iloc[0]["subset_features"])
    all_feats_union.extend(feats)

needed_pairs = sorted(set((v, lon) for (v, lon, _, _) in all_feats_union))
pc_dict = {
    (v, lon): load_pc_latdepth(v, lon, N_MODES, EOF_DIR, mode=MODE, member_id=member_id)
    for (v, lon) in needed_pairs
}

common_years = amoc["year"].values.astype(int)
for da in pc_dict.values():
    common_years = np.intersect1d(common_years, da["year"].values.astype(int))

years = np.asarray(common_years, int)
years.sort()
y_amoc = amoc.sel(year=years).values.astype(float)
pc_np = {k: v.sel(year=years).values.astype(float) for k, v in pc_dict.items()}

print(f"✅ Common years: {years[0]}–{years[-1]} (T={len(years)})")

GLOBAL_MAXLAG_FIXED = max(lag for *_, lag in all_feats_union)
print("✅ GLOBAL_MAXLAG_FIXED:", GLOBAL_MAXLAG_FIXED)


# ============================================================
# 1) RECONSTRUCTION FIGURES
# ============================================================
k_rows = []
pred_store = {}

for k in range(1, K_MAX_USE + 1):
    row = df_best.loc[df_best["k"] == k]
    if len(row) != 1:
        print(f"⚠️ skipping k={k} (not unique)")
        continue

    feats_k = parse_features_string(row.iloc[0]["subset_features"])
    if len(feats_k) == 0:
        print(f"⚠️ skipping k={k} (no feats parsed)")
        continue

    years_used, X_dt, Y_dt, idx_tr, idx_te, (ay, by), used = build_XY_for_feats(
        feats_k, years, y_amoc, pc_np, global_maxlag=GLOBAL_MAXLAG_FIXED
    )

    best_alpha = float(ALPHA_FIXED)
    mdl = make_pipeline(StandardScaler(), Ridge(alpha=float(best_alpha)))

    Xtr, Ytr = X_dt[idx_tr], Y_dt[idx_tr]
    Xte, Yte = X_dt[idx_te], Y_dt[idx_te]

    m_tr = finite_rows_mask(Ytr, Xtr)
    m_te = finite_rows_mask(Yte, Xte)

    if m_tr.sum() < 3 or m_te.sum() < 2:
        print(f"⚠️ k={k}: skipping scoring (finite rows train={m_tr.sum()} test={m_te.sum()})")
        r2_tr = np.nan
        r2_te = np.nan
        c_tr = np.nan
        c_te = np.nan
        pred_tr = np.full(len(idx_tr), np.nan, float)
        pred_te = np.full(len(idx_te), np.nan, float)
    else:
        mdl.fit(Xtr[m_tr], Ytr[m_tr])

        pred_tr = mdl.predict(Xtr)
        pred_te = mdl.predict(Xte)

        r2_tr = safe_r2(Ytr[m_tr], pred_tr[m_tr])
        r2_te = safe_r2(Yte[m_te], pred_te[m_te])
        c_tr = pearson_corr(Ytr[m_tr], pred_tr[m_tr])
        c_te = pearson_corr(Yte[m_te], pred_te[m_te])

    k_rows.append({
        "k": k,
        "alpha": best_alpha,
        "train_r2": r2_tr,
        "test_r2": r2_te,
        "train_corr": c_tr,
        "test_corr": c_te,
        "features": " | ".join([f"{v} {lon} EOF{m0+1} lag{lag}" for (v, lon, m0, lag) in feats_k]),
        "maxlag": int(max(l for *_, l in feats_k)),
        "global_maxlag": int(GLOBAL_MAXLAG_FIXED),
        "years_used_start": int(years_used[0]),
        "years_used_end": int(years_used[-1]),
    })

    y_pred_all = np.full(len(years), np.nan, float)
    y_pred_all[used[idx_tr]] = pred_tr
    y_pred_all[used[idx_te]] = pred_te

    if DETREND_Y:
        y_true_plot = y_amoc - (ay * years.astype(float) + by)
        y_pred_plot = y_pred_all
        ylab = "AMOC (detrended; TRAIN-fit)"
    else:
        y_true_plot = y_amoc
        y_pred_plot = y_pred_all
        ylab = "AMOC"

    pred_store[k] = dict(
        years=years.copy(),
        y_true=y_true_plot.copy(),
        y_pred=y_pred_plot.copy(),
        ylab=ylab
    )

    info = (
        f"alpha={best_alpha:g}\n"
        f"Train R²={r2_tr:.3f}  r={c_tr:.3f}\n"
        f"Test  R²={r2_te:.3f}  r={c_te:.3f}\n\n"
        "Features:\n" + "\n".join([f"{v} {lon} EOF{m0+1} lag{lag}" for (v, lon, m0, lag) in feats_k])
    )

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(years, y_true_plot, label="True AMOC")
    ax.plot(years, y_pred_plot, label="Predicted AMOC")
    ax.axvline(TRAIN_END_YEAR, linestyle="--", label="Split (EOF train end)")
    ax.set_title(f"{MODEL} {TARGET} — reconstruction (k={k})")
    ax.set_xlabel("Year")
    ax.set_ylabel(ylab)
    format_year_axis(ax, years, step=10)
    ax.grid(True, alpha=0.3)

    ax.text(
        0.02, 0.02, info,
        transform=ax.transAxes,
        ha="left", va="bottom",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.9, edgecolor="0.2")
    )

    ax.legend(loc="lower left", bbox_to_anchor=(0.28, 0.02), frameon=True)
    fig.tight_layout()

    outbase = os.path.join(OUTDIR, f"01_reconstruction_k{k}")
    savefig(outbase)
    plt.close()
    print("✅ Saved:", outbase + ".png/.pdf")

    print(f"✅ k={k} | alpha={best_alpha:g} | test_R2={r2_te:.4f} | years_used={years_used[0]}–{years_used[-1]}")

df_k = pd.DataFrame(k_rows).sort_values("k").reset_index(drop=True)
df_k.to_csv(os.path.join(OUTDIR, "reconstruction_summary_by_k.csv"), index=False)
print("✅ Saved: reconstruction_summary_by_k.csv")


# ============================================================
# 1b) COMBINED RECONSTRUCTION FIGURE
# ============================================================
if len(pred_store) >= 1:
    ks_plot = sorted(pred_store.keys())

    years0 = pred_store[ks_plot[0]]["years"]
    y_true0 = pred_store[ks_plot[0]]["y_true"]
    ylab0 = pred_store[ks_plot[0]]["ylab"]

    fig, ax = plt.subplots(figsize=(13, 5.2))
    ax.plot(years0, y_true0, linewidth=1.5, label="True AMOC")

    for k in ks_plot:
        ax.plot(
            pred_store[k]["years"],
            pred_store[k]["y_pred"],
            linewidth=1.0,
            label=f"Pred (k={k})"
        )

    ax.axvline(TRAIN_END_YEAR, linestyle="--", label="Split (EOF train end)")
    ax.set_title(f"{MODEL} {TARGET} — reconstructions for k={ks_plot[0]}..{ks_plot[-1]}")
    ax.set_xlabel("Year")
    ax.set_ylabel(ylab0)
    format_year_axis(ax, years0, step=10)
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2, frameon=True)
    fig.tight_layout()

    outbase = os.path.join(OUTDIR, f"01_reconstruction_ALL_k1to{max(ks_plot)}")
    savefig(outbase)
    plt.close()
    print("✅ Saved:", outbase + ".png/.pdf")


# ============================================================
# 1c) PAPER-READY COMBINED RECONSTRUCTION FIGURE
# ============================================================
if len(pred_store) >= 1:
    TARGET_K = 3

    if TARGET_K in pred_store:
        ks_plot = sorted(pred_store.keys())
        years0 = pred_store[ks_plot[0]]["years"]
        y_true0 = pred_store[ks_plot[0]]["y_true"]
        ylab0 = pred_store[ks_plot[0]]["ylab"]

        fig, ax = plt.subplots(figsize=(13, 6.5))

        color_map = {
            1: "tab:orange",
            2: "tab:green",
            3: "tab:blue"
        }

        for k in ks_plot:
            if k not in color_map:
                ax.plot(
                    pred_store[k]["years"],
                    pred_store[k]["y_pred"],
                    linewidth=0.8,
                    alpha=0.2,
                    color="gray",
                    label=f"Pred (k={k}) (additional features)"
                )

        for k in sorted(color_map.keys()):
            if k not in pred_store:
                continue

            row = df_k.loc[df_k["k"] == k].iloc[0]
            feats_str = str(row["features"]).replace(" | ", ", ")

            if k == TARGET_K:
                label_str = (
                    f"Pred (k={k}): {feats_str}\n"
                    f"Train R²={row['train_r2']:.2f}, r={row['train_corr']:.2f}  |  "
                    f"Test R²={row['test_r2']:.2f}, r={row['test_corr']:.2f}"
                )
                lw = 2.0
                alpha = 1.0
            else:
                label_str = f"Pred (k={k}): {feats_str}"
                lw = 1.5
                alpha = 0.85

            ax.plot(
                pred_store[k]["years"],
                pred_store[k]["y_pred"],
                linewidth=lw,
                alpha=alpha,
                color=color_map[k],
                label=label_str
            )

        ax.plot(
            years0,
            y_true0,
            linewidth=2.5,
            color="black",
            label="True AMOC"
        )

        ax.axvline(TRAIN_END_YEAR, linestyle="--", color="black", alpha=0.7, label="Split (Train/Test)")
        ax.set_title(f"{MODEL} {TARGET} — AMOC Reconstruction (Highlighting k={TARGET_K})")
        ax.set_xlabel("Year")
        ax.set_ylabel(ylab0)
        format_year_axis(ax, years0, step=10)
        ax.grid(True, alpha=0.3)

        leg = ax.legend(
            loc="lower left",
            bbox_to_anchor=(0.02, 0.02),
            frameon=True,
            framealpha=0.9,
            facecolor="white",
            edgecolor="0.8",
            fontsize=10
        )

        handles = getattr(leg, "legend_handles", getattr(leg, "legendHandles", []))
        for text, handle in zip(leg.get_texts(), handles):
            if hasattr(handle, "get_color"):
                text.set_color(handle.get_color())

        fig.tight_layout()

        outbase = os.path.join(OUTDIR, f"01_reconstruction_PAPER_READY_v2.1_legend_inside_k{TARGET_K}")
        savefig(outbase)
        plt.close()
        print(f"✅ Saved paper-ready plot v2.1 (legend inside): {outbase}.png/.pdf")


# ============================================================
# 2) MODE/PC plots (deduped)
# ============================================================
mode_sig_to_ks = {}
mode_sig_to_modes = {}

for k in range(1, K_MAX_USE + 1):
    row = df_best.loc[df_best["k"] == k]
    if len(row) != 1:
        print(f"⚠️ skipping MODE/PC k={k} (not unique)")
        continue

    feats_k = parse_features_string(row.iloc[0]["subset_features"])
    if len(feats_k) == 0:
        print(f"⚠️ skipping MODE/PC k={k} (no feats parsed)")
        continue

    uniq_modes_k = []
    seen = set()
    for (v, lon, m0, lag) in feats_k:
        key = (v, lon, int(m0))
        if key not in seen:
            uniq_modes_k.append(key)
            seen.add(key)
    uniq_modes_k = uniq_modes_k[:MAX_MODE_PANELS]

    if len(uniq_modes_k) == 0:
        print(f"⚠️ skipping MODE/PC k={k} (no unique modes)")
        continue

    signature = tuple(uniq_modes_k)

    mode_sig_to_ks.setdefault(signature, []).append(k)
    mode_sig_to_modes.setdefault(signature, uniq_modes_k)

for signature, ks in mode_sig_to_ks.items():
    uniq_modes = mode_sig_to_modes[signature]
    n_pan = len(uniq_modes)

    fig, axes = plt.subplots(nrows=n_pan, ncols=2, figsize=(14, 3.8 * n_pan))
    if n_pan == 1:
        axes = np.array([axes])

    for i, (v, lon, m0) in enumerate(uniq_modes):
        mode1 = int(m0) + 1
        ax_map = axes[i, 0]
        ax_pc = axes[i, 1]

        Z, depth, lat = load_eof_map_latdepth(v, lon, EOF_DIR, mode_index_1based=mode1)
        vmax = np.nanmax(np.abs(Z))
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0
        norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

        if depth[0] > depth[-1]:
            depth_plot = depth[::-1]
            Z_plot = Z[::-1, :]
        else:
            depth_plot = depth
            Z_plot = Z

        pcm = ax_map.pcolormesh(lat, depth_plot, Z_plot, shading="auto", cmap="RdBu_r", norm=norm)
        ax_map.invert_yaxis()
        ax_map.set_xlabel("Latitude")
        ax_map.set_ylabel("Depth")
        ax_map.set_title(f"{v} {lon} | EOF mode {mode1}")
        cb = fig.colorbar(pcm, ax=ax_map, fraction=0.046, pad=0.04)
        cb.set_label("EOF amplitude")

        def lat_formatter(x, pos):
            if x < 0:
                return f"{abs(int(x))}°S"
            elif x > 0:
                return f"{int(x)}°N"
            else:
                return "0°"

        ax_map.xaxis.set_major_locator(mticker.MultipleLocator(20))
        ax_map.xaxis.set_major_formatter(mticker.FuncFormatter(lat_formatter))

        PCfull = load_pc_latdepth_with_members(v, lon, N_MODES, EOF_DIR).sel(year=years)
        train_mask_full = years <= TRAIN_END_YEAR

        if "member" in PCfull.dims:
            pcs = PCfull.isel(mode=int(m0))
            if set(pcs.dims) != {"year", "member"}:
                raise ValueError(f"Unexpected PC dims for spaghetti: {pcs.dims}")

            pcs = pcs.transpose("year", "member")
            pcs_np = pcs.values.astype(float)

            if DETREND_X:
                pcs_dt = []
                for j in range(pcs_np.shape[1]):
                    s_dt, _ = detrend_with_train_fit(years, pcs_np[:, j], train_mask_full)
                    pcs_dt.append(s_dt)
                pcs_dt = np.vstack(pcs_dt).T
                pcs_mean = np.nanmean(pcs_dt, axis=1)

                ax_pc.plot(years, pcs_dt, linewidth=PC_SPAGHETTI_LW, alpha=PC_SPAGHETTI_ALPHA, color="grey")
                ax_pc.plot(years, pcs_mean, linewidth=PC_MEAN_LW, color="black", label="PC mean")
                ax_pc.set_ylabel("PC (members detrended; TRAIN-fit)")
            else:
                pcs_mean = np.nanmean(pcs_np, axis=1)
                ax_pc.plot(years, pcs_np, linewidth=PC_SPAGHETTI_LW, alpha=PC_SPAGHETTI_ALPHA, color="grey")
                ax_pc.plot(years, pcs_mean, linewidth=PC_MEAN_LW, color="black", label="PC mean")
                ax_pc.set_ylabel("PC (members)")
        else:
            pc1d = PCfull.isel(mode=int(m0)).values.astype(float)
            if DETREND_X:
                pc1d, _ = detrend_with_train_fit(years, pc1d, train_mask_full)
                ax_pc.set_ylabel("PC (detrended; TRAIN-fit)")
            else:
                ax_pc.set_ylabel("PC")
            ax_pc.plot(years, pc1d, linewidth=PC_MEAN_LW, color="black", label="PC")

        ax_pc.axvline(TRAIN_END_YEAR, linestyle="--", alpha=0.8)
        ax_pc.set_xlabel("Year")
        ax_pc.set_title(f"{v} {lon} | PC mode {mode1}")
        ax_pc.grid(True, alpha=0.3)
        format_year_axis(ax_pc, years, step=20)
        ax_pc.legend(loc="upper left", frameon=True)

    fig.suptitle(
        f"{MODEL} {TARGET} — modes used (k={ks}, unique panels={n_pan})",
        y=1.02
    )
    plt.tight_layout()

    ktag = "k" + "-".join(str(x) for x in ks)
    outbase = os.path.join(OUTDIR, f"02_EOFmap_and_PC_{ktag}_panels{n_pan}")
    savefig(outbase)
    plt.close()
    print("✅ Saved:", outbase + ".png/.pdf")


# ============================================================
# 3) TWO-PANEL EWS plots (grouped)
# ============================================================
mode_signature_to_ks = {}
mode_signature_to_feats = {}

for k in range(1, K_MAX_USE + 1):
    row = df_best.loc[df_best["k"] == k]
    if len(row) != 1:
        print(f"⚠️ skipping EWS k={k} (not unique)")
        continue

    feats_k = parse_features_string(row.iloc[0]["subset_features"])
    if len(feats_k) == 0:
        print(f"⚠️ skipping EWS k={k} (no feats parsed)")
        continue

    uniq_modes_k = []
    seen = set()
    for (v, lon, m0, lag) in feats_k:
        key = (v, lon, int(m0))
        if key not in seen:
            uniq_modes_k.append(key)
            seen.add(key)

    uniq_modes_k = uniq_modes_k[:MAX_MODE_PANELS]

    if len(uniq_modes_k) == 0:
        print(f"⚠️ skipping EWS k={k} (no unique modes)")
        continue

    maxlag_k = int(max(lag for *_, lag in feats_k))
    signature = (tuple(uniq_modes_k), maxlag_k)

    mode_signature_to_ks.setdefault(signature, []).append(k)
    mode_signature_to_feats.setdefault(signature, feats_k)

for (modes_tuple, maxlag_k), ks in mode_signature_to_ks.items():
    feats_rep = mode_signature_to_feats[(modes_tuple, maxlag_k)]
    uniq_modes = list(modes_tuple)
    n_pan = len(uniq_modes)

    years_used, X_dt, Y_dt, idx_tr, idx_te, (ay, by), used = build_XY_for_feats(
        feats_rep, years, y_amoc, pc_np, global_maxlag=GLOBAL_MAXLAG_FIXED
    )
    train_mask_used = years_used <= TRAIN_END_YEAR

    amoc_ar1_series = Y_dt.copy()
    amoc_var_series = Y_dt.copy()

    if VAR_STANDARDIZE_FOR_PLOTTING:
        amoc_var_series, _ = standardize_with_train_fit(amoc_var_series, train_mask_used)

    amoc_ar1_years, amoc_ar1_vals = rolling_ar1(
        amoc_ar1_series,
        years_used,
        window=AR1_WINDOW_YEARS,
        min_valid=AR1_MIN_VALID
    )

    amoc_var_years, amoc_var_vals = rolling_variance(
        amoc_var_series,
        years_used,
        window=VAR_WINDOW_YEARS,
        min_valid=VAR_MIN_VALID,
        ddof=VAR_DDOF
    )

    fig, axes = plt.subplots(
        nrows=2, ncols=1, figsize=(12, 8.2), sharex=True,
        gridspec_kw={"hspace": 0.12}
    )
    ax1, ax2 = axes

    for (v, lon, m0) in uniq_modes:
        pc_series = pc_np[(v, lon)][used, int(m0)]

        if DETREND_X:
            pc_series, _ = detrend_with_train_fit(years_used, pc_series, train_mask_used)

        pc_ar1_years, pc_ar1_vals = rolling_ar1(
            pc_series,
            years_used,
            window=AR1_WINDOW_YEARS,
            min_valid=AR1_MIN_VALID
        )

        pc_var_series = pc_series.copy()
        if VAR_STANDARDIZE_FOR_PLOTTING:
            pc_var_series, _ = standardize_with_train_fit(pc_var_series, train_mask_used)

        pc_var_years, pc_var_vals = rolling_variance(
            pc_var_series,
            years_used,
            window=VAR_WINDOW_YEARS,
            min_valid=VAR_MIN_VALID,
            ddof=VAR_DDOF
        )

        label = f"{v} {lon} EOF{int(m0)+1}"

        ax1.plot(
            pc_ar1_years, pc_ar1_vals,
            linewidth=LINEWIDTH_PC,
            label=label
        )
        ax2.plot(
            pc_var_years, pc_var_vals,
            linewidth=LINEWIDTH_PC,
            label=label
        )

    ax1.plot(
        amoc_ar1_years, amoc_ar1_vals,
        linewidth=LINEWIDTH_AMOC,
        linestyle="--",
        label="AMOC"
    )
    ax2.plot(
        amoc_var_years, amoc_var_vals,
        linewidth=LINEWIDTH_AMOC,
        linestyle="--",
        label="AMOC"
    )

    for ax in axes:
        ax.axvline(TRAIN_END_YEAR, linestyle=":", alpha=0.9)
        ax.grid(True, alpha=0.3)
        format_year_axis(ax, years_used, step=10)

    ax1.set_ylabel("Lag-1 autocorrelation")
    ax1.set_ylim(-0.5, 1.0)
    ax1.set_title(
        f"{MODEL} {TARGET} — Early-warning indicators\n"
        f"k={ks} | unique panels={n_pan} | maxlag={maxlag_k}"
    )

    if VAR_STANDARDIZE_FOR_PLOTTING:
        ax2.set_ylabel("Rolling variance\n(standardized)")
    else:
        ax2.set_ylabel("Rolling variance")
    ax2.set_xlabel("Year")

    handles, labels = ax1.get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc="upper center",
        ncol=min(3, max(2, len(labels))),
        frameon=True,
        bbox_to_anchor=(0.5, 0.995)
    )

    fig.tight_layout(rect=[0, 0, 1, 0.94])

    ktag = "k" + "-".join(str(x) for x in ks)
    suffix = "stdvar" if VAR_STANDARDIZE_FOR_PLOTTING else "rawvar"
    outbase = os.path.join(
        OUTDIR,
        f"03_EWS_twopanel_{ktag}_panels{n_pan}_maxlag{maxlag_k}_ARW{AR1_WINDOW_YEARS}_VW{VAR_WINDOW_YEARS}_{suffix}"
    )
    savefig(outbase)
    plt.close()
    print("✅ Saved:", outbase + ".png/.pdf")


# ============================================================
# 5) CONTRIBUTION PLOT
# ============================================================
df_k = df_k.sort_values("k").reset_index(drop=True)
Kc = len(df_k)

if Kc >= 1:
    r2s = df_k["test_r2"].values.astype(float)

    deltas = np.empty_like(r2s)
    deltas[0] = r2s[0]
    for i in range(1, len(r2s)):
        deltas[i] = r2s[i] - r2s[i - 1]

    df_contrib = df_k[["k", "alpha", "test_r2"]].copy()
    df_contrib["delta_test_r2"] = deltas
    contrib_csv = os.path.join(OUTDIR, "05_r2_contribution_by_k.csv")
    df_contrib.to_csv(contrib_csv, index=False)
    print("✅ Saved:", contrib_csv)

    step_colors = [
        "tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple",
        "tab:brown", "tab:pink", "tab:olive", "tab:cyan"
    ]

    x = np.arange(1, Kc + 1)
    fig, ax = plt.subplots(figsize=(10.5, 4.8))

    for i in range(Kc):
        if i > 0 and deltas[i] < 0:
            ax.bar(x[i], r2s[i], color="0.6", edgecolor="0.1")
            ax.text(
                x[i], r2s[i],
                f"{r2s[i]:.3f}",
                ha="center",
                va="bottom" if r2s[i] >= 0 else "top",
                fontsize=7
            )
            continue

        bottom = 0.0
        for j in range(i + 1):
            h = deltas[j]
            if h <= 0:
                continue
            col = step_colors[j % len(step_colors)]
            ax.bar(x[i], h, bottom=bottom, color=col, edgecolor="0.1")
            bottom += h

        ax.text(
            x[i], r2s[i],
            f"{r2s[i]:.3f}",
            ha="center",
            va="bottom" if r2s[i] >= 0 else "top",
            fontsize=7
        )

    ax.axhline(0, linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels([f"k={int(k)}" for k in df_k["k"].values])
    ax.set_xlabel("Model size (k features)")
    ax.set_ylabel("TEST R² (cumulative)")
    ax.set_title(f"{MODEL} {TARGET} | Stacked contributions to TEST R² (k=1..{Kc})")
    ax.grid(True, axis="y", alpha=0.25)

    handles = []
    labels = []
    for j in range(min(Kc, len(step_colors))):
        handles.append(plt.Rectangle((0, 0), 1, 1, color=step_colors[j], ec="0.2"))
        labels.append(f"Contribution step {j+1}")
    handles.append(plt.Rectangle((0, 0), 1, 1, color="0.6", ec="0.2"))
    labels.append("Decrease (grey bar)")
    ax.legend(handles, labels, loc="best", frameon=True)

    fig.tight_layout()
    outbase = os.path.join(OUTDIR, "05_r2_contribution_waterfall_by_k_stacked")
    savefig(outbase)
    plt.close()
    print("✅ Saved:", outbase + ".png/.pdf")


print("\n✅ Final clean figure script completed.")